# Учебное исследование города Кореновска — BlocksNet 1.0.0a9

Исследуем **только город Кореновск, Краснодарский край, Россия**. Граница района или всего
городского поселения не подходит. Квартал здесь — участок, ограниченный дорогами, водой и другими
преградами; это аналитическая единица, а не кадастровый квартал.

Ноутбук подготовлен **без выполнения ячеек, загрузки исследовательских данных и расчётов**.
Вы запускаете каждую ячейку самостоятельно через **Shift+Enter**. Сначала прочитайте пояснение,
затем выполните код под ним и посмотрите результат. Не используйте «Run All» при первом знакомстве.

**Маршрут:** окружение → граница → дороги и вода → кварталы → **СТОП и обсуждение метрик** →
здания, зоны и школы → показатели → готовый граф и время → проверка и сохранение.

**Проблема с графом на этапе подготовки (30.10.2026):** не удалось подтвердить конкретный доступный
готовый граф с полным покрытием города. Поэтому источник не выдуман, автоматической замены нет,
а транспортные расчёты заблокированы. Это не мешает подготовить кварталы и независимые показатели.
HTML-дашборд — следующий отдельный этап, здесь он не создаётся.


## 0. Открытие на macOS и источники

1. Откройте **папку проекта** в VS Code: File → Open Folder.
2. При необходимости установите расширения **Python** и **Jupyter** от Microsoft.
3. Откройте этот файл. Справа сверху нажмите **Select Kernel → Python Environments** и выберите
   `.venv/bin/python` из этого проекта. Не выбирайте системный Python.
4. Начните с раздела **1**, выполняйте ячейки по очереди. После раздела **4** остановитесь.

Если библиотек не хватает, выполните **самостоятельно** в терминале VS Code, открытом в корне проекта:

```bash
source .venv/bin/activate
python -m pip install -e '.[research]'
```

Команда использует группу `research` в существующем `pyproject.toml`; отдельного requirements.txt нет.
BlocksNet фиксирован на `1.0.0a9`, он требует IduEdu `0.4.1`. Не обновляйте их до последних версий.
PyTorch нужен из-за импорта внутри нарезки кварталов; обучение модели здесь не используется.

**Особенность этого Mac:** текущее окружение — x86_64, в нём установлен PyTorch 2.2.2.
Для macOS Intel официальные готовые пакеты новых PyTorch недоступны
([обсуждение разработчиков PyTorch](https://github.com/pytorch/pytorch/issues/137281)).
Поэтому `pyproject.toml` сохраняет 2.2.2 для этой архитектуры, а не требует недоступное обновление.
Связка старого PyTorch с NumPy 2 может выдавать предупреждение инициализации NumPy.
Этот ноутбук не вызывает преобразования NumPy↔тензоры и не использует нейросетевой классификатор;
работоспособность импорта в вашем окружении должна подтвердить первая ячейка.
Если она завершилась **исключением**, остановитесь и покажите сообщение: не понижайте NumPy
ниже 2, поскольку это противоречит требованиям BlocksNet 1.0.0a9.

Если установка завершилась ошибкой, сохраните её текст и остановитесь — не запускайте последующие ячейки.

Источники и воспроизводимость:

- [Pipeline BlocksNet](https://github.com/aimclub/blocksnet/blob/main/examples/pipeline.ipynb) —
  последовательность подготовки, показатели и примеры. Просмотрен 30.10.2026; ветка main меняется.
- [Исходный код BlocksNet](https://github.com/aimclub/blocksnet) и **локальная установленная 1.0.0a9** —
  сигнатуры функций проверены чтением исходников, без импорта и выполнения исследовательского кода.
- [OpenStreetMap](https://www.openstreetmap.org/copyright) — будущая загрузка границы и объектов через
  Nominatim/Overpass с OSMnx. Указывайте © OpenStreetMap contributors, ODbL в последующих публикациях.
- [Global Urban Street Networks GraphML](https://doi.org/10.7910/DVN/KA5HJ3) — проверявшийся **каталог-кандидат**,
  не подтверждённый граф Кореновска. Публикация архива не доказывает покрытие города.

В отличие от старого примера, в 1.0.0a9 функция `area_accessibility` находится в
`blocksnet.analysis.network.accessibility`. Её смысл — среднее время, взвешенное по площади кварталов.


## 1. Окружение, папки и правила повторного запуска

Первая ячейка только проверяет версии, импортирует библиотеки и создаёт рабочие папки.
Она ничего не загружает и не считает для города. Все пути определяются относительно проекта,
независимо от того, запущен Jupyter из корня или из `examples/`.

Рабочие файлы появятся **после вашего запуска** в `examples/data/korenovsk/`:
`input/` — ваши файлы, `raw/` — исходные слои, `processed/` — кварталы и обработанные слои,
`results/` — показатели, таблицы и изображения. GeoJSON сохраняется в WGS84 (EPSG:4326),
а площади и длины считаются после перевода в местную систему координат с метрами.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import sys
import platform
from datetime import datetime, timezone
from enum import Enum
from urllib.request import Request, urlopen

assert version("blocksnet") == "1.0.0a9", "Нужно выбрать окружение с BlocksNet 1.0.0a9"
assert version("iduedu") == "0.4.1", "BlocksNet 1.0.0a9 требует IduEdu 0.4.1"

import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
from pyproj import CRS
from IPython.display import display
# Проверяем импорт нарезки заранее: до любой загрузки данных. Кварталы здесь не считаются.
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file() and (p / "my_package").is_dir()), None)
assert ROOT is not None, "Откройте папку проекта и выберите её рабочей папкой"
BASE = ROOT / "examples" / "data" / "korenovsk"
INPUT, RAW, PROCESSED, RESULTS = [BASE / name for name in ("input", "raw", "processed", "results")]
for folder in (INPUT, RAW, PROCESSED, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)
VERSIONS = {name: version(name) for name in
            ("blocksnet", "iduedu", "geopandas", "osmnx", "networkx", "numpy", "pandas", "torch")}
print("Python:", sys.executable, "; архитектура:", platform.machine())
display(pd.Series(VERSIONS, name="Версия"))
print("Папка исследования:", BASE)


### 1.1. Настройки — сначала всё закрыто

`False` означает «выключено», `True` — «включено». Для будущей загрузки OSM вы самостоятельно
поменяете `ALLOW_OSM_DOWNLOADS` на `True` и ещё раз выполните эту ячейку.
Сначала разрешите только получение границы, посмотрите её и подтвердите в разделе 2.1.
Скачивание готового графа имеет отдельный выключатель.

`REFRESH_RAW = False` сохраняет уже загруженный снимок OSM. `True` означает явное обновление.
После обновления исходных данных результаты автоматически перестают считаться совместимыми.
Проверяйте соответствие границы, даже если файл уже был на диске.


In [ ]:
CITY_QUERY = "Кореновск, Краснодарский край, Россия"
ALLOW_OSM_DOWNLOADS = False
REFRESH_RAW = False
FORCE_RECALCULATE = False

# Для своих точных границ положите city_boundary.geojson в папку input.
BOUNDARY_INPUT = INPUT / "city_boundary.geojson"
# Не выбираем номер результата геокодера за вас.
NOMINATIM_RESULT = None  # при неоднозначном поиске укажите номер только после просмотра

# Предположения pipeline: это учебные оценки, не сведения о реальном населении города.
DEFAULT_FLOORS = 1
LIVING_AREA_COEFFICIENT = 0.9
LIVING_DEMAND_M2 = 30

# Защитные подтверждения сбрасываются при повторном выполнении этой ячейки.
BOUNDARY_APPROVED_HASH = ""
METRICS_APPROVED_HASH = ""
ESTIMATES_ACCEPTED = False
GRAPH_COVERAGE_APPROVED_HASH = ""
ALLOW_GRAPH_DOWNLOAD = False

ox.settings.use_cache = True
ox.settings.cache_folder = str(BASE / "osm_cache")
ox.settings.requests_timeout = 180
ox.settings.log_console = False


### 1.2. Служебные функции хранения

Эта ячейка только определяет функции. GeoJSON сопровождается JSON-паспортом: параметры,
версии, источник, дата и контрольная сумма файла. Контрольная сумма — «отпечаток» содержимого:
если входной файл изменился, прежний результат не подхватывается молча.
Отсутствие объектов после успешного ответа и сбой загрузки — разные ситуации: сетевые ошибки не скрываются.


In [ ]:
NOTEBOOK_REVISION = "korenovsk-build-1"

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True, default=str).encode()).hexdigest()

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def frame_hash(frame):
    return digest(json.loads(frame.to_crs(4326).to_json(drop_id=False)))

def json_write(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")

def scalar(value):
    if isinstance(value, Enum):
        return value.value
    if isinstance(value, (list, tuple, dict, set)):
        return json.dumps(list(value) if isinstance(value, set) else value, ensure_ascii=False, default=str)
    return value

def save_layer(frame, path, details=None):
    out = frame.copy().to_crs(4326)
    # Стандартный GeoJSON не допускает Infinity/NaN. Их причины сохраняются в отдельных флагах.
    for column in out.columns:
        if column == out.geometry.name:
            continue
        if out[column].dtype == object:
            out[column] = out[column].map(scalar)
    out = out.replace([np.inf, -np.inf], np.nan)
    payload = out.to_json(drop_id=True, na="null")
    Path(path).write_text(payload, encoding="utf-8")
    json_write(Path(path).with_suffix(".meta.json"), {
        "created_utc": datetime.now(timezone.utc).isoformat(), "versions": VERSIONS,
        "revision": NOTEBOOK_REVISION, "sha256": file_hash(path), "rows": len(out),
        "storage_crs": "EPSG:4326", "details": details or {},
    })

def read_layer(path, crs=4326):
    # Ручной разбор позволяет надёжно прочитать и пустой FeatureCollection.
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    if payload["features"]:
        return gpd.GeoDataFrame.from_features(payload["features"], crs=4326).to_crs(crs)
    return gpd.GeoDataFrame(geometry=[], crs=4326).to_crs(crs)

def cached_layer(path, builder, params, crs=4326, refresh=False, companions=()):
    path = Path(path)
    meta = path.with_suffix(".meta.json")
    signature = digest({"params": params, "versions": VERSIONS, "revision": NOTEBOOK_REVISION})
    if path.exists() and meta.exists() and not refresh and all(Path(p).exists() for p in companions):
        passport = json.loads(meta.read_text(encoding="utf-8"))
        if (passport["details"].get("signature") == signature and passport["sha256"] == file_hash(path)):
            print("Из готового файла:", path.name)
            return read_layer(path, crs)
        print("Изменились входные данные/параметры:", path.name)
    result = builder()
    save_layer(result, path, {"signature": signature, "parameters": params})
    print("Сохранено:", path.name)
    return read_layer(path, crs)

def check_boundary():
    if BOUNDARY_APPROVED_HASH != frame_hash(boundary):
        raise RuntimeError("СТОП: сначала подтвердите границу именно города в разделе 2.1")

def check_metrics():
    check_boundary()
    current = digest({"blocks": file_hash(PROCESSED / "blocks.geojson"), "boundary": frame_hash(boundary)})
    if METRICS_APPROVED_HASH != current:
        raise RuntimeError("СТОП ПОСЛЕ КВАРТАЛОВ. Сначала обсудите метрики и выполните раздел 5.1")

def fetch_features(tags, geometry_types):
    check_boundary()
    if not ALLOW_OSM_DOWNLOADS:
        raise RuntimeError("Файла нет: загрузка OSM выключена в разделе 1.1")
    from osmnx._errors import InsufficientResponseError
    try:
        features = ox.features_from_polygon(boundary.to_crs(4326).union_all(), tags=tags)
    except InsufficientResponseError as exc:
        # Не маскируем тайм-аут, повреждённый ответ и другие ошибки под пустые данные.
        if "No matching features" not in str(exc):
            raise
        print("Успешный запрос: подходящих объектов не найдено", tags)
        return gpd.GeoDataFrame(geometry=[], crs=4326)
    features = features.reset_index()
    features = features[features.geom_type.isin(geometry_types)].copy()
    features = features[features.geometry.notna() & ~features.geometry.is_empty].copy()
    features["geometry"] = features.geometry.make_valid()
    features = features[features.geom_type.isin(geometry_types)].reset_index(drop=True)
    # Для линий важно пересечение: длинная дорога может пересекать город, имея середину за его границей.
    features = features.to_crs(LOCAL_CRS)
    features = features.loc[features.intersects(boundary.union_all())].copy()
    return features

def get_osm_layer(name, tags, geometry_types):
    def builder():
        features = fetch_features(tags, geometry_types)
        if name in ("buildings", "schools"):
            # Не обрезаем площадь здания на границе; сохраняем целые объекты по внутренней точке.
            points = features.representative_point()
            features = features.loc[points.intersects(boundary.union_all())].copy()
        elif not features.empty:
            features = gpd.clip(features, boundary)
            features = features[features.geom_type.isin(geometry_types)].copy()
        return features.reset_index(drop=True)
    return cached_layer(RAW / f"{name}.geojson", builder,
                        {"source": "OpenStreetMap/Nominatim/Overpass via OSMnx",
                         "boundary": frame_hash(boundary), "tags": tags, "types": geometry_types},
                        crs=LOCAL_CRS, refresh=REFRESH_RAW)

def restore_blocks(frame):
    assert "block_id" in frame and frame.block_id.is_unique, "Нужны уникальные block_id"
    return frame.set_index("block_id", drop=False).rename_axis(None).sort_index()


## 2. Получить и проверить границу города

Эта ячейка при разрешённой загрузке обращается **только за границей**, а не за всеми объектами.
Если у вас есть точная граница города в GeoJSON, положите её в `input/city_boundary.geojson` —
загрузка геокодером не потребуется. Файл должен иметь координаты WGS84 (долгота, широта).

Геокодер может вернуть границу городского поселения вместо города. Название в ответе само по себе
не является доказательством. Проверьте контур по карте и источнику; если пришёл район или поселение,
**не подтверждайте его**. Обсудим выбор точной границы отдельно. Прямоугольник вокруг центра
не заменяет границу города.


In [ ]:
def build_boundary():
    if BOUNDARY_INPUT.exists():
        candidate = read_layer(BOUNDARY_INPUT)
        candidate["boundary_source"] = "Пользовательский city_boundary.geojson"
    else:
        if not ALLOW_OSM_DOWNLOADS:
            raise RuntimeError("Разрешите получение границы в разделе 1.1 или добавьте свой файл")
        candidate = ox.geocode_to_gdf(CITY_QUERY, which_result=NOMINATIM_RESULT)
        candidate["boundary_source"] = "Nominatim: " + CITY_QUERY
    assert len(candidate) == 1, "Нужен один объект границы города (допустим MultiPolygon)"
    assert candidate.geom_type.isin(["Polygon", "MultiPolygon"]).all(), "Нужна полигональная граница"
    assert candidate.geometry.is_valid.all(), "Граница некорректна: требуется ручная проверка"
    return candidate

boundary = cached_layer(
    RAW / "city_boundary.geojson", build_boundary,
    {"query": CITY_QUERY, "result": NOMINATIM_RESULT,
     "user_file": file_hash(BOUNDARY_INPUT) if BOUNDARY_INPUT.exists() else None}, refresh=REFRESH_RAW)
LOCAL_CRS = boundary.estimate_utm_crs()
assert LOCAL_CRS is not None and LOCAL_CRS.is_projected
boundary = boundary.to_crs(LOCAL_CRS)
print("Местная система координат:", LOCAL_CRS)
print("Площадь выбранного контура, км²:", round(boundary.area.sum() / 1_000_000, 2))
display(boundary.drop(columns="geometry").T)
display(boundary.to_crs(4326).explore(tooltip=False))
print("Отпечаток границы:", frame_hash(boundary))


### 2.1. Ручное подтверждение границы

Сравните контур с территорией **самого города**. Если уверенности нет, остановитесь и покажите
результат предыдущей ячейки. Когда граница проверена, впишите ровно `ПОДТВЕРЖДАЮ ГОРОД` вместо
пустых кавычек ниже. Подтверждение связано с конкретным контуром и не действует для другого файла.


In [ ]:
BOUNDARY_CONFIRMATION = ""
if BOUNDARY_CONFIRMATION != "ПОДТВЕРЖДАЮ ГОРОД":
    raise RuntimeError("Граница ещё не подтверждена. Не переходите к загрузке дорог")
BOUNDARY_APPROVED_HASH = frame_hash(boundary)
print("Подтверждён контур города. Можно подготовить кварталы")


## 3. Исходные слои для кварталов

Нужны линии дорог и железных дорог, а также реки и водоёмы. Это геометрия для разрезания территории,
**не маршрутный граф**. Ячейка загружает каждый слой только при отсутствии совместимого файла.
Не прерывайте её повторным нажатием запуска. При сетевой ошибке посмотрите сообщение и повторите
ячейку позже: уже успешно сохранённые слои будут прочитаны с диска.


In [ ]:
check_boundary()
LINE_TYPES = ["LineString", "MultiLineString"]
POLYGON_TYPES = ["Polygon", "MultiPolygon"]
ROAD_TAGS = {"highway": ["construction", "living_street", "motorway", "motorway_link", "pedestrian",
                        "primary", "primary_link", "residential", "road", "secondary", "secondary_link",
                        "tertiary", "tertiary_link", "track", "trunk", "trunk_link", "unclassified"],
             "service": ["living_street", "emergency_access"]}
WATER_TAGS = {"riverbank": True, "reservoir": True, "basin": True, "dock": True, "canal": True,
              "pond": True, "natural": ["water", "bay"], "waterway": ["river", "canal", "ditch"],
              "landuse": "basin", "water": "lake"}
roads = get_osm_layer("roads", ROAD_TAGS, LINE_TYPES)
railways = get_osm_layer("railways", {"railway": "rail"}, LINE_TYPES)
water = get_osm_layer("water", WATER_TAGS, LINE_TYPES + POLYGON_TYPES)
assert not roads.empty, "Нет дорог — нарезку пока не выполняйте"
display(pd.Series({"дороги": len(roads), "железные дороги": len(railways), "водные объекты": len(water)},
                  name="Число объектов"))
ax = boundary.boundary.plot(color="black", figsize=(10, 10))
roads.plot(ax=ax, color="grey", linewidth=0.5)
if not railways.empty:
    railways.plot(ax=ax, color="red", linewidth=0.7)
if not water.empty:
    water.plot(ax=ax, color="blue", linewidth=0.5)
ax.set_title("Проверьте линии, по которым будут выделены кварталы")
ax.set_axis_off()
plt.show()


## 4. Подготовить кварталы и сохранить результат

Ячейка использует `preprocess_urban_objects` и `cut_urban_blocks` из BlocksNet 1.0.0a9.
Обучение и классификатор не запускаются. Водные полигоны вычитаются из территории — поэтому
сумма площадей кварталов может быть меньше площади городской границы.
Ничего мелкого автоматически не удаляем: необычно маленькие и большие участки нужно осмотреть.
Номер `block_id` хранится в GeoJSON и затем используется во всех таблицах.


In [ ]:
check_boundary()
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

def build_blocks():
    lines, polygons = preprocess_urban_objects(roads.copy(), railways.copy(), water.copy())
    result = cut_urban_blocks(boundary[["geometry"]].copy(), lines, polygons)
    result = result.explode(index_parts=False).reset_index(drop=True)
    result = result[result.geom_type == "Polygon"].copy()
    assert len(result) > 0 and result.geometry.is_valid.all()
    # Стабильная сортировка при неизменном входном снимке.
    result["sort_x"] = result.representative_point().x
    result["sort_y"] = result.representative_point().y
    result = result.sort_values(["sort_x", "sort_y"]).reset_index(drop=True)
    result["block_id"] = result.index.astype(int)
    result["site_area"] = result.area
    return result[["block_id", "site_area", "geometry"]]

blocks = restore_blocks(cached_layer(
    PROCESSED / "blocks.geojson", build_blocks,
    {"boundary": frame_hash(boundary), "inputs": {name: file_hash(RAW / f"{name}.geojson")
     for name in ("roads", "railways", "water")}, "method": "cut_urban_blocks; classifier=None"},
    crs=LOCAL_CRS, refresh=FORCE_RECALCULATE))
blocks["site_area"] = blocks.area
assert blocks.block_id.is_unique and (blocks.site_area > 0).all()
assert blocks.geometry.is_valid.all()
city_geometry = boundary.union_all()
assert blocks.geometry.difference(city_geometry).area.sum() < 1, "Кварталы выходят за контур города"
overlap_m2 = blocks.area.sum() - blocks.union_all().area
assert overlap_m2 < 1, "Кварталы перекрываются: требуется проверка"
print("Кварталов:", len(blocks), "; площадь кварталов, га:", round(blocks.area.sum() / 10_000, 2))
display(blocks[["block_id", "site_area"]].describe())
display(blocks.to_crs(4326).explore(column="block_id", categorical=True, tooltip=["block_id", "site_area"],
                                 legend=False))
print("Кварталы сохранены:", PROCESSED / "blocks.geojson")


### 4.1. Зафиксированные ограничения автоматической нарезки

По итогам диагностики, **сообщённым пользователем**: получено **403 квартала**, геометрия валидна,
перекрытий нет. Выявлено **15 потенциальных артефактов около железной дороги** и
**36 крупных кварталов площадью более 10 га**. Агент эти числа повторно не рассчитывал.

Ручная корректировка и повторная нарезка **не выполняются**: продолжаем с текущими 403 кварталами.
Потенциальный артефакт не означает доказанную ошибку, а большая площадь сама по себе не является
дефектом. Узкие фрагменты и крупные кварталы влияют на распределение и сравнимость показателей:
из-за небольшого знаменателя плотности отдельных фрагментов могут быть нестабильны, а средние
характеристики крупных кварталов могут скрывать неоднородность застройки. Учитываем это в выводах.
Номера этих кварталов здесь не выдумываются. Сводка хранится также в метаданных ноутбука и будет
включена в `quality_report.json` **только после вашего запуска раздела 12**.


### 4.2. Прочитать уже проверенные кварталы без повторной нарезки

Если прежнее ядро работает и переменная `blocks` доступна, этот шаг можно пропустить.
Если VS Code перезапускался: выполните раздел **1** (импорты, настройки и служебные функции),
**2** (чтение сохранённой границы) и **2.1** (подтверждение границы), затем следующую ячейку.
Разделы **3** и ячейку нарезки **4** повторно запускать не нужно.
Сначала оставьте загрузки и обновления выключенными. При ошибке чтения остановитесь и покажите её.

Следующая ячейка только читает `blocks.geojson` и сверяет контрольную сумму с паспортом.
Она не исправляет геометрию, не пересчитывает площади и не запускает нарезку.


In [ ]:
check_boundary()
reviewed_blocks_path = PROCESSED / "blocks.geojson"
assert reviewed_blocks_path.is_file(), "Не найден сохранённый blocks.geojson; не запускайте нарезку автоматически"
reviewed_meta = json.loads(reviewed_blocks_path.with_suffix(".meta.json").read_text(encoding="utf-8"))
assert file_hash(reviewed_blocks_path) == reviewed_meta["sha256"], "Файл отличается от паспорта: сначала уточните причину"
blocks = restore_blocks(read_layer(reviewed_blocks_path, crs=LOCAL_CRS))
print("Прочитаны сохранённые кварталы:", len(blocks), "; повторная нарезка не выполнялась")


# СТОП: кварталы готовы — сначала обсудим доступные метрики

**Здесь закончите первый запуск.** Проверьте карту, особенно реку, железную дорогу, окраины,
необычно большие кварталы и узкие фрагменты. Граница должна относиться к городу, а не поселению.
Напишите в чате: «Кварталы готовы. Какие метрики доступны для моих данных?» и сообщите число кварталов,
при необходимости приложите карту или ошибку.

Следующие ячейки написаны заранее, но заблокированы. Даже если VS Code продолжит «Run All» после
ошибки, каждая исследовательская ячейка после остановки отдельно проверяет разрешение.
При повторном запуске с новым контуром или новыми кварталами подтверждение нужно заново.


In [ ]:
# Намеренная остановка, а не неисправность. Эту ячейку не нужно «исправлять».
raise RuntimeError("СТОП: кварталы подготовлены. Перед показателями обсудите доступные метрики в чате")


## 5. Справочник показателей и разрешение продолжения

Весь перечень из pipeline предусмотрен ниже. Дополнительно показана плотность населения.
Площади считаются в м², площадь квартала в гектарах = м² / 10 000.

| Показатель | Что означает | Что требуется |
|---|---|---|
| Назначение территории | Жилая, промышленная, рекреационная и другие зоны | Полигоны `landuse`; OSM не является официальным генпланом |
| Население и жители/га | Число жителей и его отношение к площади квартала | Данные населения или явно принятая оценка по жилой площади |
| Пятно застройки | Сумма площадей оснований зданий | Контуры зданий |
| Поэтажная, жилая и нежилая площадь | Суммарные площади помещений | Этажность и назначение; при пропусках — предположения |
| FSI | Поэтажная площадь / площадь квартала | Площади и этажность |
| GSI | Площадь пятен / площадь квартала | Контуры зданий |
| MXI | Жилая площадь / поэтажная площадь | Оценка жилой части |
| L | Поэтажная площадь / площадь пятен, условная средняя этажность | Площади и этажность |
| OSR | (Площадь квартала − пятна) / поэтажная площадь | Площади и этажность |
| share_living, share_non_living | Жилая и нежилая площадь / площадь пятен | Площади помещений; это **не проценты**, сумма может превышать 1 |
| Spacematrix, «Стрелка» | Условные типы застройки по плотности, этажности и смешанности | FSI, L, MXI |
| Количество школ и вместимость | Число объектов и мест по кварталам | OSM и сведения о вместимости; иначе модельная подстановка |
| Время между кварталами | Кратчайший путь по готовому автомобильному графу | Подтверждённый граф с временными весами |
| provision_strong / provision_weak | Доля спроса, покрытого местами в пределах порога / с учётом более дальних мест | Население, места, матрица времени и явно выбранный порог |
| area_accessibility | Среднее время до кварталов, взвешенное по их площади | Матрица времени и площади кварталов |

Нулевое население, неизвестное население и отсутствие жилых зданий — разные вещи.
Для пустых кварталов часть коэффициентов математически не определена: в GeoJSON будет `null`
и поясняющий флаг. Не будем превращать деление на ноль в «нулевую плотность».


### Все показатели pipeline простыми словами

Обозначим: **S** — площадь квартала, **B** — сумма пятен зданий, **F** — сумма поэтажных площадей,
**R** — жилая площадь, **N** — нежилая площадь. Эти площади выражены в **м²**.

| Поле | Что показывает и как считается | Как понимать |
|---|---|---|
| `site_area` | S, площадь квартала | Для гектаров делим на 10 000 |
| `footprint_area` | B, сумма площадей оснований зданий | Сколько места занимают здания на земле; это сумма, не геометрическое объединение |
| `number_of_floors` | Этажность отдельного здания | Пропуски в учебном сценарии заменяются 1 этажом; не складываем этажность зданий как характеристику квартала |
| `build_floor_area` | F, сумма площадей всех этажей | При отсутствии сведений: площадь основания × этажность |
| `living_area`, `non_living_area` | R и N, жилые и нежилые помещения | Для жилых зданий доля жилой площади модельно принята 90%; N = F − R |
| `is_living` | Признак жилого здания по OSM | Неизвестное назначение не доказывает, что здание нежилое |
| `population` | Оценка числа жителей | В этом сценарии по жилой площади при допущении 30 м²/чел.; не перепись |
| `buildings_count` | Число зданий, отнесённых к кварталу | Проверяем также здания, не попавшие ни в один квартал |
| `land_use`, `land_use_share` | Назначение с наибольшей долей площади пересечения и эта доля | Теги OSM не заменяют официальный генплан; неизвестное назначение сохраняется |
| `schools_count`, `capacity` | Количество школ и суммарное число мест | При пропуске вместимости библиотека подставляет типовое значение, не фактическую вместимость |

**Коэффициенты застройки из pipeline** безразмерные — это отношения площадей:

| Коэффициент | Формула | Простой смысл |
|---|---|---|
| **FSI** | F / S | Интенсивность использования территории: сколько площади этажей приходится на площадь квартала; может быть больше 1 |
| **GSI** | B / S | Доля земли под зданиями: 0.3 означает около 30% |
| **MXI** | R / F | Доля жилой площади во всей площади этажей: 0.8 означает около 80% |
| **L** | F / B | Средняя этажность, взвешенная по площади оснований зданий |
| **OSR** | (S − B) / F | Открытая площадь на единицу площади этажей; это не доля озеленения и не оценка качества благоустройства |
| `share_living` | R / B | Жилая площадь относительно пятна застройки |
| `share_non_living` | N / B | Нежилая площадь относительно пятна застройки |

Последние два показателя **не являются процентными долями от общей площади** и могут превышать 1.
При согласованных площадях их сумма равна L. В BlocksNet 1.0.0a9 GSI и MXI ограничиваются диапазоном
0–1, L — снизу единицей; исходные аномалии площадей дополнительно проверяются.
Для незастроенных кварталов некоторые отношения не определены, поэтому возможны пустые значения.

**Дополнение нашего ноутбука:** `population_density_per_ha` = население / (S / 10 000), жителей/га.
Это плотность относительно всей площади квартала, не только жилых участков.

**Морфотипы:**

- **Spacematrix** группирует кварталы по FSI, L и MXI, затем описывает группы по этажности,
  плотности и жилому/смешанному/нежилому использованию. Это относительная классификация данного
  набора кварталов, не оценка «хорошо/плохо». Сохраняются тип, номер группы и характеристики групп.
- **«Стрелка»** назначает тип застройки по правилам и порогам FSI, L и MXI: например,
  индивидуальная жилая или микрорайонная. Это модельная классификация, не историческая экспертиза.

### Транспортные показатели: объясняем, но сейчас не считаем

- **Матрица времени**: строка — квартал отправления, столбец — квартал назначения, значение —
  минуты по графу выбранного способа передвижения. Прямое расстояние её не заменяет.
- **Спрос `demand`**: сколько мест в школах требуется жителям квартала по модельному числу мест
  на 1000 жителей; результат округляется вверх. Это не фактический список школьников.
- **Конкурентная обеспеченность** распределяет ограниченные места между кварталами:
  одно место нельзя одновременно отдать нескольким жителям. `demand_within` — покрытый спрос
  в пределах порога времени, `demand_without` — покрытый за пределами порога в рассматриваемом
  расширенном диапазоне, `demand_left` — непокрытый спрос. `capacity_within`, `capacity_without`
  и `capacity_left` описывают использование и остаток мест **в квартале расположения школы**.
- **`provision_strong`** = `demand_within / demand`: доля спроса, покрытого в пределах порога.
- **`provision_weak`** = (`demand_within + demand_without`) / `demand`: доля покрытого спроса,
  включая более дальние места. При стандартном `max_depth=1` расширения поиска нет и показатели
  совпадают. На кварталах без спроса отношение не определено.
- **Таблица распределения `links_df`**: откуда, куда и сколько единиц спроса назначено; это
  модель распределения мест, не наблюдаемые поездки жителей.
- **`area_accessibility`** = сумма(время до квартала × площадь этого квартала) / сумма площадей:
  среднее время до территории города с весами по площади кварталов. Единица — минуты;
  меньшее время означает лучшую доступность в этом смысле. Это не число школ поблизости.

**Без подтверждённого графа эти транспортные результаты отсутствуют. Не записываем вместо них
нули и не делаем выводов о транспортном или территориальном дефиците школ.** Количество школ
и вместимость по кварталам можно подготовить без графа, обеспеченность с учётом времени — нельзя.
В pipeline также упоминаются расстояния по прямой и граф соседства как другие способы описать
связи кварталов; это не транспортные показатели и не замена выбранному способу расчёта.


### 5.1. Выполнить только после обсуждения в чате

После обсуждения впишите `МЕТРИКИ ОБСУЖДЕНЫ` в следующей ячейке. Подтверждение нужно при каждом
новом сеансе. Ячейку с намеренной остановкой в разделе 4 при продолжении пропустите.
Если меняете параметры в разделе 1.1, повторите также подтверждение границы.


### Порядок продолжения практики без готового графа

Сначала прочитайте объяснения выше. Затем запускайте **ячейки под указанными заголовками**, а не
по счётчикам `[1]`, `[2]`: счётчик обозначает порядок выполнения и может измениться.

1. Если ядро перезапускалось, восстановите окружение и кварталы по разделу **4.2**. Исходные
   кварталы оставляем без правок. Если ядро сохранилось, можно продолжать с текущими переменными.
2. **5.1:** впишите `МЕТРИКИ ОБСУЖДЕНЫ` в `METRICS_CONFIRMATION` и выполните ячейку.
   Это открывает расчёты для текущего файла кварталов.
3. Если зданий, зон и школ ещё нет на диске, установите **только** `ALLOW_OSM_DOWNLOADS = True`
   в отдельной новой ячейке и выполните её. Это разрешение для ваших последующих запусков,
   само по себе оно ничего не скачивает. Не перезапускайте всю ячейку настроек 1.1: она сбросит
   подтверждения. Оставьте `REFRESH_RAW=False` и `FORCE_RECALCULATE=False`.
4. **6:** загрузка/чтение зон и назначение использования кварталов.
5. **7:** загрузка/чтение зданий и просмотр долей пропусков — сначала оцените качество данных.
6. **7.1:** если согласны с учебными предположениями, впишите `ПРИНИМАЮ УЧЕБНЫЕ ОЦЕНКИ`
   в `ESTIMATES_CONFIRMATION`, затем выполните ячейку. **7.2:** суммирование по кварталам.
7. **8:** загрузка/чтение школ, проверка карты и дублей. **8.1:** при необходимости заполните
   `EXCLUDE_SCHOOL_IDS`, впишите `СПИСОК ШКОЛ ПРОВЕРЕН` и выполните ячейку.
8. **9:** коэффициенты застройки. **9.1:** морфотипы Spacematrix и «Стрелка».
9. **Разделы 10–11 пропустите целиком:** готового графа нет, доступность и обеспеченность по времени
   не рассчитываем. Защиты не снимаем, транспортные значения не подставляем.
10. **12:** сводная таблица, графики, экспорт GeoJSON/CSV и отчёт качества с ограничениями нарезки.
    При пропуске 10–11 будет честный статус «не рассчитано» без фиктивных транспортных значений.
    В **12.1** напишите выводы по реально полученным результатам. Правила повторного запуска — в **13**.

Если какая-то ячейка завершилась ошибкой, остановитесь на ней и покажите сообщение. Если школ
не оказалось в OSM или список пока не подтверждён, не делайте из этого вывод об отсутствии школ:
можно завершить 9 и 12 без школьных показателей, явно указав их отсутствие.

Для сдачи практики сохраните ноутбук с выводами после вашего выполнения, итоговые GeoJSON/CSV,
отчёт качества и изображения. В выводах укажите 403 квартала, 15 потенциальных артефактов у ЖД,
36 крупных кварталов, модельные допущения о зданиях/населении/вместимости и отсутствие графа.
Отдельного текста задания преподавателя в проекте не предоставлено: этот порядок покрывает
согласованный pipeline, но дополнительные требования нужно сверить с формулировкой задания.


In [ ]:
METRICS_CONFIRMATION = ""
check_boundary()
if METRICS_CONFIRMATION != "МЕТРИКИ ОБСУЖДЕНЫ":
    raise RuntimeError("Не продолжайте до обсуждения метрик")
METRICS_APPROVED_HASH = digest({"blocks": file_hash(PROCESSED / "blocks.geojson"),
                                "boundary": frame_hash(boundary)})
print("Расчёты разрешены для текущей версии кварталов")


## 6. Функциональные зоны

Загружаются полигоны `landuse` из OSM. Это наблюдаемые теги, а не утверждённое градостроительное
зонирование. Непокрытые кварталы сохраняются с неизвестным назначением. В словаре ниже —
правила из pipeline; их можно обсудить и изменить. Смена словаря обновит результат.


In [ ]:
check_metrics()
from blocksnet.enums import LandUse
from blocksnet.blocks.assignment import assign_land_use

zones = get_osm_layer("functional_zones", {"landuse": True}, POLYGON_TYPES)
RULES = {
    "commercial": LandUse.BUSINESS, "industrial": LandUse.INDUSTRIAL,
    "cemetery": LandUse.SPECIAL, "garages": LandUse.INDUSTRIAL,
    "residential": LandUse.RESIDENTIAL, "retail": LandUse.BUSINESS,
    "grass": LandUse.RECREATION, "farmland": LandUse.AGRICULTURE,
    "construction": LandUse.SPECIAL, "brownfield": LandUse.INDUSTRIAL,
    "forest": LandUse.RECREATION, "recreation_ground": LandUse.RECREATION,
    "religious": LandUse.SPECIAL, "flowerbed": LandUse.RECREATION,
    "military": LandUse.SPECIAL, "landfill": LandUse.TRANSPORT,
}

def build_land_use():
    usable = zones.loc[zones.get("landuse", pd.Series(index=zones.index, dtype=object)).isin(RULES)].copy()
    if usable.empty:
        result = blocks[["geometry"]].copy()
        result["land_use"] = None
        result["share"] = np.nan
    else:
        usable = usable[["geometry", "landuse"]].rename(columns={"landuse": "functional_zone"})
        result = assign_land_use(blocks[["geometry"]].copy(), usable, RULES)
    result["block_id"] = result.index
    return result

land_use = restore_blocks(cached_layer(PROCESSED / "land_use.geojson", build_land_use,
    {"blocks": file_hash(PROCESSED / "blocks.geojson"), "zones": file_hash(RAW / "functional_zones.geojson"),
     "rules": {k: v.value for k, v in RULES.items()}}, crs=LOCAL_CRS, refresh=FORCE_RECALCULATE))
display(land_use.land_use.fillna("Нет данных").value_counts())
if land_use.land_use.notna().any():
    land_use.plot(column="land_use", categorical=True, legend=True, figsize=(10, 8),
                  missing_kwds={"color": "lightgrey", "label": "Нет данных"}).set_axis_off()
    plt.show()


## 7. Здания, этажность и модельная оценка населения

Будут загружены контуры зданий. `building:levels` — указанное в OSM число этажей.
Для пропусков pipeline использует 1 этаж; жилой площади отводится 90% общей площади,
население оценивается по 30 м² жилой площади на человека. Эти числа **не подтверждены для Кореновска**.
Квартиры, пустующие дома и реальная заселённость из этих формул неизвестны.

Тег `building=yes` не сообщает, жилое ли здание. Такие объекты отдельно отмечаются как неизвестные;
в учебном сценарии они не добавляют население. Поэтому оценка может существенно ошибаться.
Сначала посмотрите доли пропусков. Если оценки неприемлемы, остановитесь и найдём дополнительные данные.


In [ ]:
check_metrics()
buildings_raw = get_osm_layer("buildings", {"building": True}, POLYGON_TYPES)
assert not buildings_raw.empty, "Здания не найдены: показатели застройки пока недоступны"
raw_floors = pd.to_numeric(buildings_raw.get("building:levels",
                           pd.Series(np.nan, index=buildings_raw.index)), errors="coerce")
known_residential_tags = ["residential", "house", "apartments", "detached", "terrace", "dormitory",
                          "semidetached_house", "bungalow"]
unknown_building = buildings_raw.building.isna() | buildings_raw.building.isin(["yes"])
display(pd.Series({"Зданий": len(buildings_raw),
                   "Этажность отсутствует/нечисловая/неположительная": int((raw_floors.isna() | (raw_floors <= 0)).sum()),
                   "Неизвестно, жилое ли здание": int(unknown_building.sum())}))


### 7.1. Принять учебные предположения и подготовить здания

Только после обсуждения напишите `ПРИНИМАЮ УЧЕБНЫЕ ОЦЕНКИ`. Все такие результаты будут подписаны
как модельные. Население не подгоняется к численности района или поселения.


In [ ]:
check_metrics()
ESTIMATES_CONFIRMATION = ""
if ESTIMATES_CONFIRMATION != "ПРИНИМАЮ УЧЕБНЫЕ ОЦЕНКИ":
    raise RuntimeError("Сначала обсудите и примите допущения об этажности, жилой площади и населении")
ESTIMATES_ACCEPTED = True
from blocksnet.preprocessing.imputing import impute_buildings

def build_imputed_buildings():
    prepared = buildings_raw[["geometry"]].copy()
    prepared["is_living"] = buildings_raw.building.isin(known_residential_tags)
    prepared["number_of_floors"] = raw_floors.where(raw_floors > 0)
    result = impute_buildings(prepared, default_number_of_floors=DEFAULT_FLOORS,
                              living_area_coefficient=LIVING_AREA_COEFFICIENT,
                              default_living_demand=LIVING_DEMAND_M2)
    result["population"] = result.population.fillna(0)
    result["floors_estimated"] = raw_floors.isna() | (raw_floors <= 0)
    result["residential_unknown"] = unknown_building
    result["population_source"] = "Модель по жилой площади, не перепись"
    result["object_id"] = result.index
    return result

building_params = {"input": file_hash(RAW / "buildings.geojson"), "floors": DEFAULT_FLOORS,
                   "living_coefficient": LIVING_AREA_COEFFICIENT, "m2_per_person": LIVING_DEMAND_M2,
                   "living_tags": known_residential_tags}
buildings = cached_layer(PROCESSED / "buildings.geojson", build_imputed_buildings, building_params,
                         crs=LOCAL_CRS, refresh=FORCE_RECALCULATE)
print("Оценка населения по загруженным зданиям:", int(buildings.population.sum()))
print("Это модель, а не установленная численность жителей города")


### 7.2. Суммировать характеристики зданий по кварталам

Каждое здание относится к кварталу по внутренней точке. Двойные назначения на общей границе
проверяются до суммирования. Отдельно сохраняются здания, не попавшие ни в один квартал:
их нельзя молча потерять при оценке качества результата.


In [ ]:
check_metrics()
if not ESTIMATES_ACCEPTED:
    raise RuntimeError("Учебные оценки не приняты")
from blocksnet.blocks.aggregation import aggregate_objects
BUILDING_COLUMNS = ["footprint_area", "build_floor_area", "living_area", "non_living_area", "population"]

def check_unique_assignment(objects):
    pts = objects[["geometry"]].copy()
    pts.geometry = pts.representative_point()
    hits = pts.sjoin(blocks[["geometry"]], how="inner", predicate="intersects")
    if hits.index.duplicated().any():
        raise ValueError("Некоторые объекты относятся к нескольким кварталам: нужна ручная проверка")

def build_buildings_blocks():
    check_unique_assignment(buildings)
    result, outside = aggregate_objects(blocks[["geometry"]].copy(), buildings[["geometry"] + BUILDING_COLUMNS])
    result = result.rename(columns={"count": "buildings_count"})
    result["block_id"] = result.index
    result["site_area"] = result.area
    result["population_density_per_ha"] = result.population / (result.site_area / 10_000)
    result["population_estimated"] = True
    result["no_buildings_in_osm"] = result.buildings_count == 0
    save_layer(outside, RESULTS / "buildings_unassigned.geojson", {"reason": "Не попали в кварталы"})
    print("Зданий вне кварталов:", len(outside))
    return result

buildings_blocks = restore_blocks(cached_layer(PROCESSED / "buildings_blocks.geojson", build_buildings_blocks,
    {"blocks": file_hash(PROCESSED / "blocks.geojson"), "buildings": file_hash(PROCESSED / "buildings.geojson")},
    crs=LOCAL_CRS, refresh=FORCE_RECALCULATE, companions=[RESULTS / "buildings_unassigned.geojson"]))
display(buildings_blocks.drop(columns="geometry").head())
ax = buildings_blocks.plot(column="population_density_per_ha", legend=True, figsize=(10, 8))
ax.set_title("Модельная плотность населения, жителей/га")
ax.set_axis_off()
plt.show()


## 8. Школы и вместимость

Сохраняем исходные школьные объекты OSM. Узел школы, её территория и отдельное здание могут
дублировать один объект. Сначала посмотрите таблицу и карту. Автоматическое удаление по одному
названию ненадёжно, поэтому ниже есть список `EXCLUDE_SCHOOL_IDS` для явно проверенных дублей.
В него вписываются номера строк `school_id`. После проверки нужно подтвердить список.


In [ ]:
check_metrics()
schools_raw = get_osm_layer("schools", {"amenity": "school"},
                            ["Point", "MultiPoint", "Polygon", "MultiPolygon"])
schools_review = schools_raw.copy().reset_index(drop=True)
schools_review["school_id"] = schools_review.index
review_columns = [c for c in ["school_id", "element", "id", "name", "addr:street", "addr:housenumber", "capacity"]
                  if c in schools_review]
display(schools_review[review_columns])
if not schools_review.empty:
    display(schools_review.to_crs(4326).explore(tooltip=review_columns))
else:
    print("OSM не вернул школ. Это не доказывает отсутствие школ в городе")


### 8.1. Проверенный список школ и модельные места

Заполните исключения при наличии дублей, затем напишите `СПИСОК ШКОЛ ПРОВЕРЕН`.
Нечисловая или отсутствующая вместимость заменяется минимальной типовой вместимостью из конфигурации
BlocksNet — так делает pipeline. Это учебная модель, **не реальное число мест в школе**.
Для выводов о реальном дефиците понадобятся подтверждённые вместимости и данные населения.


In [ ]:
check_metrics()
if not ESTIMATES_ACCEPTED:
    raise RuntimeError("Оценки должны быть приняты в разделе 7.1")
EXCLUDE_SCHOOL_IDS = []
SCHOOLS_CONFIRMATION = ""
if SCHOOLS_CONFIRMATION != "СПИСОК ШКОЛ ПРОВЕРЕН":
    raise RuntimeError("Проверьте дубли школ и подтвердите список")
from blocksnet.preprocessing.imputing import impute_services
from blocksnet.config import service_types_config

SCHOOL_CONFIG = service_types_config["school"]
print("Параметры библиотеки, не утверждённые нормативы Кореновска:", SCHOOL_CONFIG)

schools = schools_review.loc[~schools_review.school_id.isin(EXCLUDE_SCHOOL_IDS)].copy()
if schools.empty:
    raise RuntimeError("Нет проверенных школ: обеспеченность не рассчитываем как достоверный ноль")

def build_schools():
    prepared = schools[["geometry"]].copy()
    capacity = pd.to_numeric(schools.get("capacity", pd.Series(np.nan, index=schools.index)), errors="coerce")
    prepared["capacity"] = capacity.where(capacity > 0)
    estimated = prepared.capacity.isna()
    result = impute_services(prepared, "school")
    result["capacity_estimated"] = estimated
    result["school_id"] = schools.school_id
    result["name"] = schools.get("name", pd.Series("Без названия", index=schools.index))
    return result

schools = cached_layer(PROCESSED / "schools.geojson", build_schools,
    {"input": file_hash(RAW / "schools.geojson"), "excluded_ids": EXCLUDE_SCHOOL_IDS,
     "config": SCHOOL_CONFIG, "capacity_method": "impute_services school"},
    crs=LOCAL_CRS, refresh=FORCE_RECALCULATE)
check_unique_assignment(schools)

def build_schools_blocks():
    result, outside = aggregate_objects(blocks[["geometry"]].copy(), schools[["geometry", "capacity"]])
    result = result.rename(columns={"count": "schools_count"})
    result["block_id"] = result.index
    save_layer(outside, RESULTS / "schools_unassigned.geojson")
    print("Школ вне кварталов:", len(outside))
    return result

schools_blocks = restore_blocks(cached_layer(PROCESSED / "schools_blocks.geojson", build_schools_blocks,
    {"blocks": file_hash(PROCESSED / "blocks.geojson"), "schools": file_hash(PROCESSED / "schools.geojson")},
    crs=LOCAL_CRS, refresh=FORCE_RECALCULATE, companions=[RESULTS / "schools_unassigned.geojson"]))
display(schools[["school_id", "name", "capacity", "capacity_estimated"]])
schools_blocks.plot(column="capacity", legend=True, figsize=(10, 8)).set_axis_off()
plt.title("Модельная вместимость школ, мест")
plt.show()


## 9. Показатели застройки и морфотипы (не требуют графа)

`FSI`, `GSI`, `MXI`, `L`, `OSR`, `share_living`, `share_non_living` считаются функцией из pipeline.
Для отсутствующей застройки отношения могут быть неопределены. Кроме того, реализация библиотеки
ограничивает GSI и MXI диапазоном 0–1, L — снизу единицей. Мы сохраняем отдельный флаг, когда
сумма пятен зданий превышает площадь квартала: ограничение не должно скрыть проблему данных.


In [ ]:
check_metrics()
from blocksnet.analysis.indicators import calculate_density_indicators
DENSITY_COLUMNS = ["fsi", "gsi", "mxi", "l", "osr", "share_living", "share_non_living"]

def build_density():
    base = buildings_blocks[["site_area", "footprint_area", "build_floor_area", "living_area", "non_living_area"]]
    values = calculate_density_indicators(pd.DataFrame(base))
    result = buildings_blocks.copy()
    for column in DENSITY_COLUMNS:
        result[column] = values[column]
    result["density_undefined"] = result.build_floor_area <= 0
    result["footprint_exceeds_site"] = result.footprint_area > result.site_area
    result["building_areas_estimated"] = True
    result[DENSITY_COLUMNS] = result[DENSITY_COLUMNS].replace([np.inf, -np.inf], np.nan)
    return result

density = restore_blocks(cached_layer(RESULTS / "density.geojson", build_density,
    {"buildings_blocks": file_hash(PROCESSED / "buildings_blocks.geojson")},
    crs=LOCAL_CRS, refresh=FORCE_RECALCULATE))
display(density[DENSITY_COLUMNS].describe())
print("Пятна больше площади квартала:", int(density.footprint_exceeds_site.sum()))


### 9.1. Типы застройки Spacematrix и «Стрелка»

Берём только кварталы с застройкой и конечными значениями FSI, L и MXI.
Spacematrix в pipeline использует 11 кластеров. Для маленькой выборки это невозможно, поэтому
число групп уменьшается до числа разных допустимых наблюдений; изменение явно выводится.
Номера кластеров не являются оценкой «лучше/хуже». Фиксированное случайное зерно помогает повторить расчёт.


In [ ]:
check_metrics()
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes
features = density[["l", "fsi", "mxi"]].copy()
valid = (features.fsi > 0) & np.isfinite(features).all(axis=1)
usable = pd.DataFrame(features.loc[valid])
assert len(usable) > 0, "Нет кварталов с достаточными данными для морфотипов"
N_CLUSTERS = min(11, len(usable), len(usable.drop_duplicates()))
print("Кластеров Spacematrix:", N_CLUSTERS, "; допустимых кварталов:", len(usable))

morph_params = {"density": file_hash(RESULTS / "density.geojson"), "n_clusters": N_CLUSTERS, "random_state": 42}
def build_morphotypes():
    spacematrix, clusters = get_spacematrix_morphotypes(usable, n_clusters=N_CLUSTERS, random_state=42)
    strelka = get_strelka_morphotypes(usable)
    clusters.to_csv(RESULTS / "spacematrix_clusters.csv", encoding="utf-8-sig", index_label="cluster")
    result = blocks[["block_id", "geometry"]].copy()
    result["spacematrix_cluster"] = spacematrix.cluster.reindex(result.index)
    result["spacematrix_morphotype"] = spacematrix.morphotype.reindex(result.index)
    result["strelka_morphotype"] = strelka.morphotype.reindex(result.index)
    result["morphotype_status"] = np.where(result.index.isin(usable.index), "Модельный тип", "Недостаточно данных/нет застройки")
    return result

morphotypes = restore_blocks(cached_layer(RESULTS / "morphotypes.geojson", build_morphotypes, morph_params,
                                        crs=LOCAL_CRS, refresh=FORCE_RECALCULATE,
                                        companions=[RESULTS / "spacematrix_clusters.csv"]))
for column in ("spacematrix_morphotype", "strelka_morphotype"):
    display(morphotypes[column].fillna("Не определён").value_counts())
    morphotypes.plot(column=column, categorical=True, legend=True, figsize=(12, 8),
                     missing_kwds={"color": "lightgrey", "label": "Не определён"}).set_axis_off()
    plt.title(column)
    plt.savefig(RESULTS / f"{column}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 10. Готовый граф: проблема источника и условия продолжения

**Подходящий готовый граф пока не найден и его покрытие не подтверждено.**
Проверявшийся источник: [Global Urban Street Networks GraphML, DOI 10.7910/DVN/KA5HJ3](https://doi.org/10.7910/DVN/KA5HJ3).
В доступных описаниях не удалось установить точный файл для Кореновска, границу его охвата,
направление движения и временные веса. Каталог не выбран как подтверждённый источник.
Без чтения самого графа нельзя проверить связность и привязку каждого квартала; такие данные
при подготовке ноутбука не загружались по вашему условию.

**Планируемый способ передвижения — автомобиль.** Пока источник не согласован,
не заполняйте следующий паспорт самостоятельно наугад. Мы вернёмся к этому вопросу после подготовки
кварталов. Не заменяем граф расстояниями по прямой, запросом OSMnx `graph_from_*`,
сборкой из PBF или интермодальным графом.

Ниже подготовлен код для **уже готового отдельного GraphML**: загрузка по точной ссылке с контрольной
суммой, чтение, проверка покрытия и времени. Он заработает после согласования реального источника.
Ссылка должна вести непосредственно на GraphML, не на ZIP всей страны и не на HTML-страницу.
Если подходящий граф доступен только в другом формате, сначала согласуем адаптацию.


In [ ]:
check_metrics()
GRAPH_SPEC = {
    "status": "not_found",
    "catalogue_candidate": "https://doi.org/10.7910/DVN/KA5HJ3",
    "source_page": None,      # точная страница подтверждённого источника
    "download_url": None,     # прямая HTTPS-ссылка на отдельный GraphML
    "sha256": None,           # согласованная контрольная сумма
    "published_date": None,
    "mode": "drive",
    "time_attribute": None,   # например, time_min или travel_time — только по документации источника
    "time_units": None,       # minutes или seconds
    "coverage_bbox_wgs84": None,  # [west, south, east, north] из паспорта источника
    "coverage_evidence": None,    # ссылка/описание географического покрытия
}
GRAPH_PATH = RAW / "ready_drive.graphml"
GRAPH_MAX_SNAP_METERS = 500  # учебный порог привязки внутренней точки квартала к узлу; обсудить

def require_graph_source():
    required = ["source_page", "download_url", "sha256", "published_date", "time_attribute",
                "time_units", "coverage_bbox_wgs84", "coverage_evidence"]
    if GRAPH_SPEC["status"] != "verified_source" or any(not GRAPH_SPEC[k] for k in required):
        raise RuntimeError("Готовый граф Кореновска не подтверждён. Сначала согласуйте источник в чате")
    if GRAPH_SPEC["mode"] != "drive" or GRAPH_SPEC["time_units"] not in ("minutes", "seconds"):
        raise ValueError("Нужны автомобильный граф и документированные единицы времени")
    if not str(GRAPH_SPEC["download_url"]).startswith("https://"):
        raise ValueError("Требуется точная HTTPS-ссылка на готовый GraphML")


### 10.1. Загрузка готового графа — заблокирована до подтверждения источника

Файл читается с диска, если контрольная сумма совпадает. Иначе ячейка потребует вашего явного
разрешения загрузки через `ALLOW_GRAPH_DOWNLOAD = True` в разделе 1.1. Она не строит новый граф.


In [ ]:
check_metrics()
require_graph_source()
if not GRAPH_PATH.exists():
    if not ALLOW_GRAPH_DOWNLOAD:
        raise RuntimeError("Загрузка графа выключена. Сначала согласуйте источник")
    # Загрузка выполняется только вами при запуске этой ячейки.
    temporary_path = GRAPH_PATH.with_suffix(".part")
    request = Request(GRAPH_SPEC["download_url"], headers={"User-Agent": "KorenovskStudentResearch/1.0"})
    with urlopen(request, timeout=180) as response, temporary_path.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    if file_hash(temporary_path) != GRAPH_SPEC["sha256"]:
        raise ValueError("Контрольная сумма не совпала. Файл .part не используется")
    temporary_path.replace(GRAPH_PATH)
assert file_hash(GRAPH_PATH) == GRAPH_SPEC["sha256"], "Файл графа отличается от согласованного"
json_write(RAW / "graph_source.json", GRAPH_SPEC)
print("Готовый граф сохранён:", GRAPH_PATH)


### 10.2. Проверить географическое и сетевое покрытие

Проверки не сводятся к наличию слова «Кореновск» в названии файла:

1. Охватывает ли заявленный прямоугольник покрытия весь городской контур?
2. Где реально лежат узлы и дороги? Какая часть города попадает внутрь их оболочки?
3. Как далеко каждая внутренняя точка квартала от ближайшего узла **связной части** графа?
4. Какие компоненты связности есть и какие узлы будут исключены?

Оболочка узлов — только диагностический признак, а не доказательство полноты дорог.
После таблицы нужно осмотреть карту и несколько окраин. Изолированные кварталы не объявляются доступными.
IduEdu 0.4.1 сам оставляет крупнейшую сильно связную компоненту; ниже это делается явно до проверки
привязки, чтобы исключение узлов не прошло незаметно. Это ограничение метода, а не ремонт графа.


In [ ]:
check_metrics()
require_graph_source()
assert file_hash(GRAPH_PATH) == GRAPH_SPEC["sha256"]
graph_original = ox.load_graphml(GRAPH_PATH)
assert graph_original.is_directed(), "Ожидался ориентированный автомобильный граф"
assert graph_original.number_of_edges() > 0
attribute = GRAPH_SPEC["time_attribute"]
for _, _, data in graph_original.edges(data=True):
    if attribute not in data:
        raise ValueError(f"На ребре отсутствует {attribute}. Подмена скоростью по умолчанию запрещена")
    weight = float(data[attribute])
    if not np.isfinite(weight) or weight < 0:
        raise ValueError("Недопустимое время на ребре")
    data["time_min"] = weight / 60 if GRAPH_SPEC["time_units"] == "seconds" else weight

graph_projected = ox.project_graph(graph_original, to_crs=LOCAL_CRS)
# Разные параллельные рёбра одной пары узлов: оставляем минимальное время, иначе IduEdu выбирает
# одно из рёбер при преобразовании MultiDiGraph в DiGraph без сравнения времён.
graph_simple = nx.DiGraph()
graph_simple.graph.update(graph_projected.graph)
graph_simple.graph["crs"] = int(CRS(LOCAL_CRS).to_epsg())
graph_simple.add_nodes_from(graph_projected.nodes(data=True))
for u, v, data in graph_projected.edges(data=True):
    if not graph_simple.has_edge(u, v) or data["time_min"] < graph_simple[u][v]["time_min"]:
        graph_simple.add_edge(u, v, **data)
components = sorted(nx.strongly_connected_components(graph_simple), key=len, reverse=True)
graph = graph_simple.subgraph(components[0]).copy()
# IduEdu 0.4.1 использует номера узлов как позиции в разреженной матрице.
# Нужны последовательные номера 0..N-1 в порядке добавления; исходные OSM ID сохраняем отдельно.
graph = nx.convert_node_labels_to_integers(graph, first_label=0, ordering="default", label_attribute="source_node_id")
assert list(graph.nodes) == list(range(graph.number_of_nodes()))
removed_nodes = graph_simple.number_of_nodes() - graph.number_of_nodes()

nodes = gpd.GeoDataFrame({"node_id": list(graph.nodes),
                          "source_node_id": [str(d["source_node_id"]) for _, d in graph.nodes(data=True)]},
    geometry=[Point(data["x"], data["y"]) for _, data in graph.nodes(data=True)], crs=LOCAL_CRS)
origins = blocks[["block_id", "geometry"]].copy()
origins.geometry = origins.representative_point()
snap = origins.sjoin_nearest(nodes, how="left", distance_col="snap_distance_m")
snap = snap.sort_values(["block_id", "snap_distance_m", "node_id"]).drop_duplicates("block_id")
snap["too_far"] = snap.snap_distance_m > GRAPH_MAX_SNAP_METERS
snap["node_id"] = snap.node_id.astype(str)
save_layer(snap.drop(columns="index_right", errors="ignore"), RESULTS / "graph_snap_check.geojson")

# Для просмотра в ГИС сохраняем также готовые узлы и рёбра, не только GraphML.
graph_nodes_export, graph_edges_export = ox.graph_to_gdfs(graph_projected)
save_layer(graph_nodes_export.reset_index(), RAW / "graph_nodes.geojson", GRAPH_SPEC)
save_layer(graph_edges_export.reset_index(), RAW / "graph_edges.geojson", GRAPH_SPEC)

coverage_box = box(*GRAPH_SPEC["coverage_bbox_wgs84"])
metadata_covers_city = coverage_box.covers(boundary.to_crs(4326).union_all())
hull = nodes.union_all().convex_hull
hull_share = hull.intersection(boundary.union_all()).area / boundary.area.sum()
coverage_report = {
    "metadata_bbox_covers_city": bool(metadata_covers_city),
    "nodes_original": graph_simple.number_of_nodes(), "strong_components": len(components),
    "nodes_removed_by_scc": removed_nodes, "nodes_used": graph.number_of_nodes(),
    "city_share_inside_node_hull_diagnostic_only": float(hull_share),
    "blocks_too_far": int(snap.too_far.sum()),
    "max_snap_m": float(snap.snap_distance_m.max()), "snap_limit_m": GRAPH_MAX_SNAP_METERS,
    "graph_sha256": file_hash(GRAPH_PATH), "boundary_sha256": frame_hash(boundary),
}
json_write(RESULTS / "graph_coverage_report.json", coverage_report)
display(pd.Series(coverage_report))
display(snap[["block_id", "node_id", "snap_distance_m", "too_far"]].sort_values("snap_distance_m", ascending=False).head(20))

ax = boundary.boundary.plot(color="black", figsize=(12, 10))
ox.plot_graph(graph_projected, ax=ax, node_size=0, edge_color="lightgrey", edge_linewidth=0.5,
              show=False, close=False)
nodes.plot(ax=ax, markersize=1, color="grey")
snap.plot(ax=ax, column="snap_distance_m", legend=True, markersize=12)
ax.set_title("Готовый граф и расстояния привязки кварталов, м")
plt.show()
assert metadata_covers_city, "По паспорту граф не покрывает весь город. Не считать время"
assert not snap.too_far.any(), "Есть кварталы далеко от сети. Обсудите покрытие, не повышайте порог наугад"
GRAPH_CURRENT_HASH = digest({"coverage": coverage_report, "blocks": file_hash(PROCESSED / "blocks.geojson")})


### 10.3. Подтверждение покрытия и матрица времени в минутах

После проверки карты и связности напишите `ПОКРЫТИЕ ПРОВЕРЕНО`.
Никаких фактических проверок покрытия автор ноутбука пока не выполнял — это делает только ваш запуск.

Время считается по документированным весам готового графа. Привязка к ближайшему узлу условная.
Устанавливаем `add_dist_tofrom_node=False`, чтобы IduEdu не добавлял пеший подход со скоростью 5 км/ч
к автомобильному времени. Значит, это **время по сети между узлами привязки кварталов**, не поездка
от двери до двери, без парковки и пробок, если их не учитывает исходный граф.


In [ ]:
check_metrics()
require_graph_source()
GRAPH_CONFIRMATION = ""
if GRAPH_CONFIRMATION != "ПОКРЫТИЕ ПРОВЕРЕНО":
    raise RuntimeError("Сначала проверьте покрытие по разделу 10.2")
assert coverage_report["metadata_bbox_covers_city"] and coverage_report["blocks_too_far"] == 0
GRAPH_COVERAGE_APPROVED_HASH = GRAPH_CURRENT_HASH
from blocksnet.relations import calculate_accessibility_matrix

matrix_path = RESULTS / "travel_time_minutes.csv"
matrix_meta = RESULTS / "travel_time_minutes.meta.json"
matrix_signature = digest({"coverage": GRAPH_CURRENT_HASH, "weight": "time_min", "versions": VERSIONS,
                           "connector": False, "dtype": "float64", "revision": NOTEBOOK_REVISION})
read_cached_matrix = False
if matrix_path.exists() and matrix_meta.exists() and not FORCE_RECALCULATE:
    previous = json.loads(matrix_meta.read_text(encoding="utf-8"))
    read_cached_matrix = previous.get("signature") == matrix_signature and previous.get("sha256") == file_hash(matrix_path)
if read_cached_matrix:
    travel_time = pd.read_csv(matrix_path, index_col=0)
    travel_time.index = travel_time.index.astype(int)
    travel_time.columns = travel_time.columns.astype(int)
    print("Матрица прочитана с диска")
else:
    travel_time = calculate_accessibility_matrix(blocks[["geometry"]].copy(), graph,
        weight_key="time_min", dtype=np.float64, add_dist_tofrom_node=False, max_workers=1)
    travel_time = travel_time.reindex(index=blocks.index, columns=blocks.index)
    travel_time.to_csv(matrix_path, index_label="from_block_id", encoding="utf-8-sig")
    json_write(matrix_meta, {"signature": matrix_signature, "sha256": file_hash(matrix_path),
                            "units": "minutes", "graph": GRAPH_SPEC, "coverage": coverage_report})
assert travel_time.shape == (len(blocks), len(blocks))
assert travel_time.index.equals(blocks.index) and travel_time.columns.equals(blocks.index)
assert not travel_time.isna().any().any(), "В матрице пропуски: проверить граф и номера кварталов"
assert (travel_time.to_numpy() >= 0).all()
assert np.allclose(np.diag(travel_time), 0)
print("Размер матрицы:", travel_time.shape, "; единица: минуты")
display(travel_time.iloc[:5, :5])
# Длинная таблица удобна для будущего дашборда; размер растёт как число кварталов в квадрате.
travel_time.rename_axis(index="from_block_id", columns="to_block_id").stack().rename("time_min").to_csv(
    RESULTS / "travel_time_pairs.csv", encoding="utf-8-sig")


## 11. Обеспеченность школами и доступность территории — требуют готового графа

Автомобильное время не равно пешей доступности школы. Нельзя автоматически применить порог
из конфигурации библиотеки как официальный автомобильный норматив. После обсуждения укажите
`SCHOOL_ACCESS_MINUTES` — учебный порог автомобильной доступности, и поясните его происхождение.
Спрос на 1000 жителей берётся из конфигурации, но также является модельным допущением.

`max_depth=1` повторяет стандартный вызов pipeline: учитываются места только внутри порога,
поэтому strong и weak могут совпасть. Значения на кварталах без спроса остаются неопределёнными.


In [ ]:
check_metrics()
require_graph_source()
if GRAPH_COVERAGE_APPROVED_HASH != GRAPH_CURRENT_HASH:
    raise RuntimeError("Покрытие графа не подтверждено")
SCHOOL_ACCESS_MINUTES = None
SCHOOL_THRESHOLD_REASON = ""
if SCHOOL_ACCESS_MINUTES is None or SCHOOL_ACCESS_MINUTES <= 0 or not SCHOOL_THRESHOLD_REASON:
    raise RuntimeError("Согласуйте порог автомобильной доступности школы и запишите обоснование")
assert "schools_blocks" in globals(), "Сначала подготовьте проверенные школы в разделе 8"
from blocksnet.analysis.provision import competitive_provision
from blocksnet.analysis.network.accessibility import area_accessibility

provision_input = buildings_blocks[["population"]].join(schools_blocks[["capacity"]])
assert not provision_input.isna().any().any()
provision_params = {"buildings": file_hash(PROCESSED / "buildings_blocks.geojson"),
                    "schools": file_hash(PROCESSED / "schools_blocks.geojson"),
                    "matrix": file_hash(matrix_path), "access_minutes": SCHOOL_ACCESS_MINUTES,
                    "reason": SCHOOL_THRESHOLD_REASON, "demand": int(SCHOOL_CONFIG["demand"]), "depth": 1}

def build_network_indicators():
    # CBC — решатель, поставляемый с PuLP. На macOS его ошибка выводится явно; результаты не выдумываются.
    from pulp import PULP_CBC_CMD
    assert PULP_CBC_CMD(msg=False).available(), "Решатель CBC недоступен — покажите сообщение в чате"
    provision, links = competitive_provision(provision_input, travel_time,
        accessibility=SCHOOL_ACCESS_MINUTES, demand=int(SCHOOL_CONFIG["demand"]), max_depth=1)
    links.to_csv(RESULTS / "school_allocation_links.csv", encoding="utf-8-sig")
    result = blocks[["block_id", "geometry"]].join(provision)
    # Не усредняем бесконечности или пропуски как конечное время.
    reachability = np.isfinite(travel_time).all(axis=1)
    area_values = area_accessibility(travel_time, buildings_blocks[["site_area"]])
    result["area_accessibility_min"] = area_values.area_accessibility.where(reachability)
    result["not_all_blocks_reachable"] = ~reachability
    result["travel_mode"] = "drive"
    result["model_estimates"] = True
    return result

network_results = restore_blocks(cached_layer(RESULTS / "network_indicators.geojson", build_network_indicators,
    provision_params, crs=LOCAL_CRS, refresh=FORCE_RECALCULATE,
    companions=[RESULTS / "school_allocation_links.csv"]))
display(network_results.drop(columns="geometry").head())
for column in ("provision_strong", "provision_weak", "area_accessibility_min"):
    if network_results[column].notna().any():
        network_results.plot(column=column, legend=True, figsize=(10, 8),
                             missing_kwds={"color": "lightgrey"}).set_axis_off()
        plt.title(column + (", минуты" if column.endswith("min") else ", доля"))
        plt.savefig(RESULTS / f"{column}.png", dpi=150, bbox_inches="tight")
        plt.show()


## 12. Общая таблица, графики и проверка результатов

Этот раздел можно выполнить и без графа. При отсутствии транспортных результатов в общем файле
будет явный статус, а не фиктивные нули. Таблица объединяет результаты только текущего запуска;
старые файлы от другого графа не подхватываются автоматически.

Перед выводами проверьте: правильный городской контур, отсутствие перекрытий кварталов,
доли неизвестных этажностей и назначения зданий, неподписанные школы, несоответствие пятен площади
квартала, объекты вне кварталов. Сопоставьте вручную 3–5 кварталов разных типов с картой города.
Если есть граф, дополнительно проверьте маршруты через реку, железную дорогу и окраины.


In [ ]:
check_metrics()
assert "density" in globals(), "Сначала выполните раздел 9"
summary = blocks[["block_id", "geometry"]].copy()
summary = summary.join(density.drop(columns=["geometry", "block_id"]))
if "land_use" in globals():
    summary = summary.join(land_use[["land_use", "share"]].rename(columns={"share": "land_use_share"}))
if "morphotypes" in globals():
    summary = summary.join(morphotypes[["spacematrix_cluster", "spacematrix_morphotype", "strelka_morphotype"]])
if "schools_blocks" in globals():
    summary = summary.join(schools_blocks[["schools_count", "capacity"]])
network_is_current = ("network_results" in globals() and "GRAPH_CURRENT_HASH" in globals()
                      and GRAPH_COVERAGE_APPROVED_HASH == GRAPH_CURRENT_HASH
                      and METRICS_APPROVED_HASH == digest({"blocks": file_hash(PROCESSED / "blocks.geojson"),
                                                          "boundary": frame_hash(boundary)}))
if network_is_current:
    network_columns = [c for c in network_results.columns if c not in summary.columns]
    summary = summary.join(network_results[network_columns])
    summary["network_status"] = "Рассчитано по проверенному готовому графу"
else:
    summary["network_status"] = "Не рассчитано: источник графа/покрытие/порог не подтверждены"

assert summary.block_id.is_unique and summary.index.equals(blocks.index)
assert (summary.site_area > 0).all()
assert (summary.population >= 0).all()
assert summary.geometry.is_valid.all()
quality = {
    "city": CITY_QUERY, "scope": "Только город; граница вручную подтверждена",
    "boundary_hash": frame_hash(boundary), "blocks": len(summary),
    "city_area_ha": float(boundary.area.sum() / 10_000),
    "blocks_area_ha": float(blocks.area.sum() / 10_000),
    "overlap_m2": float(blocks.area.sum() - blocks.union_all().area),
    "estimated_population_in_blocks": float(summary.population.sum()),
    "population_is_official": False,
    "automatic_cutting_limitations": {'source': 'Результаты диагностики сообщены пользователем; агент не пересчитывал', 'blocks_sha256': '99c1f0ba9026877f27b91627bcec36b66e2e0eef44f5567d92bf286a16969e3b', 'blocks_count': 403, 'geometry_valid': True, 'overlapping_pairs': 0, 'potential_rail_artifacts_count': 15, 'large_blocks_over_10ha_count': 36, 'manual_correction_performed': False, 'interpretation': 'Потенциальные артефакты не признаны доказанными ошибками; крупные кварталы не обязательно ошибочны', 'effect_on_metrics': 'Узкие фрагменты и крупные кварталы влияют на пространственное распределение и сравнимость показателей'},
    "cutting_limitations_apply_to_current_blocks": file_hash(PROCESSED / "blocks.geojson") == '99c1f0ba9026877f27b91627bcec36b66e2e0eef44f5567d92bf286a16969e3b',
    "floors_estimated_buildings": int(buildings.floors_estimated.sum()),
    "residential_unknown_buildings": int(buildings.residential_unknown.sum()),
    "undefined_density_blocks": int(density.density_undefined.sum()),
    "footprint_exceeds_site_blocks": int(density.footprint_exceeds_site.sum()),
    "unassigned_buildings": len(read_layer(RESULTS / "buildings_unassigned.geojson")),
    "school_capacity_estimated_count": (int(schools.capacity_estimated.sum())
                                         if "schools_blocks" in globals() else None),
    "unassigned_schools": (len(read_layer(RESULTS / "schools_unassigned.geojson"))
                           if "schools_blocks" in globals() else None),
    "network_calculated": bool(network_is_current), "versions": VERSIONS,
    "assumptions": {"default_floors": DEFAULT_FLOORS, "living_area_coefficient": LIVING_AREA_COEFFICIENT,
                    "living_demand_m2": LIVING_DEMAND_M2},
}
if network_is_current:
    quality["graph"] = GRAPH_SPEC
    quality["graph_coverage"] = coverage_report
    quality["school_access_minutes"] = SCHOOL_ACCESS_MINUTES
    quality["school_threshold_reason"] = SCHOOL_THRESHOLD_REASON
    demand_total = network_results.demand.sum()
    quality["school_provision_city"] = (float(network_results.demand_within.sum() / demand_total)
                                        if demand_total > 0 else None)

save_layer(summary, RESULTS / "blocks_indicators.geojson", quality)
summary.drop(columns="geometry").to_csv(RESULTS / "blocks_indicators.csv", encoding="utf-8-sig", index=False)
json_write(RESULTS / "quality_report.json", quality)
display(pd.Series({k: v for k, v in quality.items() if not isinstance(v, dict)}))
display(summary.drop(columns="geometry").head(10))

for column, label in [("population_density_per_ha", "Модельная плотность населения, жителей/га"),
                      ("fsi", "FSI: отношение поэтажной площади к площади квартала"),
                      ("gsi", "GSI: доля пятен застройки")]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    summary.plot(column=column, legend=True, ax=axes[0], missing_kwds={"color": "lightgrey"})
    axes[0].set_axis_off()
    summary[column].replace([np.inf, -np.inf], np.nan).dropna().hist(ax=axes[1], bins=20)
    axes[1].set_xlabel(label)
    axes[1].set_ylabel("Число кварталов")
    fig.suptitle(label)
    fig.tight_layout()
    fig.savefig(RESULTS / f"{column}.png", dpi=150)
    plt.show()

print("Результаты:", RESULTS)
print("Выводы о реальном дефиците услуг требуют проверки населения и вместимости школ")


### 12.1. Выводы для учебного исследования — заполните после проверки

Не делайте выводов заранее. В отдельном тексте ответьте:

1. Насколько кварталы соответствуют реальной структуре города? Где нарезка требует правки?
2. Где плотнее застройка и выше модельная плотность населения? На каких данных это основано?
3. Какие типы застройки преобладают и какие кварталы остались без определённого типа?
4. Как доля неизвестной этажности и жилого назначения влияет на выводы?
5. Если граф подтверждён: какие кварталы хуже доступны **на автомобиле**? Не переносите это на пешие поездки.
6. Если обеспеченность рассчитана: какие выводы изменятся при проверке реальной вместимости школ?
7. Какие данные ещё нужны, чтобы перейти от учебной модели к обоснованному исследованию?

Изменять текст пояснений можно двойным щелчком по Markdown-ячейке, сохранять — Shift+Enter.


## 13. Повторный запуск и передача на следующий этап

1. Сохраните ноутбук: **Cmd+S**. Сохраните также всю папку `examples/data/korenovsk/`.
2. В следующий раз откройте этот же ноутбук и выберите то же окружение `.venv`.
3. Нажмите **Restart Kernel**, затем выполните разделы 1–4 по порядку.
   Оставьте `REFRESH_RAW = False`, `FORCE_RECALCULATE = False` и загрузки выключенными:
   совместимые GeoJSON будут прочитаны с диска. При отсутствии нужного файла выполнение остановится.
4. Подтвердите прежнюю городскую границу в 2.1. После кварталов намеренную стоп-ячейку пропустите
   только если метрики уже обсуждены; заново выполните подтверждение 5.1.
5. Пройдите разделы 6–9, заново подтвердив допущения и список школ. Чтение сохранённых слоёв
   заменит повторную обработку. Раздел 10 остаётся закрытым, пока нет подтверждённого источника графа.
6. Для транспортных результатов повторите проверку покрытия и подтверждения в 10–11:
   готовая матрица прочитается из CSV, если совпадают граф, кварталы и параметры.
7. Выполните раздел 12 и сопоставьте отчёт качества с предыдущим запуском.

**Если изменили исходные данные, параметры или код расчёта:** перезапустите ядро и пройдите ячейки
сначала, не смешивайте старые переменные с новыми. Для изменённого кода увеличьте
`NOTEBOOK_REVISION` в 1.2 или включите `FORCE_RECALCULATE`. Если обновляете снимок OSM,
временно включите `ALLOW_OSM_DOWNLOADS` и `REFRESH_RAW`, затем верните их в `False`.
Не удаляйте исходные файлы для обычного повторного запуска.

**Для следующего этапа** понадобятся `blocks_indicators.geojson`, `blocks_indicators.csv`,
`quality_report.json`, изображения графиков и (если рассчитаны) `travel_time_minutes.csv`,
`travel_time_pairs.csv`, `school_allocation_links.csv`, `spacematrix_clusters.csv`, паспорт графа
и отчёт покрытия. Это входы для будущего Python-файла, создающего одну HTML-страницу.
До этого этапа не выполняем веб-разработку и публикацию.
